# 01. Огляд сирих даних
**Мета:** зрозуміти, що є у файлах Kaggle, і які процедури мають дату оголошення.

**Вхід:** `data/raw/*.csv` (Kaggle «Prozorro Ukraine Procurement 2022–2025», 12 877 960 тендерів).

**Що робимо:** розмір і структура кожного файлу (DESCRIBE); частка тендерів з `tender_start_date` за типом процедури; перший відбір за CPV-кодами.

**Результат:** у прямих закупівлях (limited) дати оголошення немає в жодному тендері, тому далі аналізуємо лише відкриті торги і каталог.

In [60]:
# Які файли і скільки вони важать
import os
import duckdb
con = duckdb.connect()
from IPython.display import Markdown

color = "purple"
path = "../data/raw/"

files = sorted([f for f in os.listdir(path) if f.endswith('.csv')])

display(Markdown(f'#### <span style="color:{color}">Які файли і скільки вони важать</span>'))

for f in files:
    print(f, round(os.path.getsize(f'{path}{f}') / 1e6, 1), 'MB')
    

#### <span style="color:purple">Які файли і скільки вони важать</span>

bidders.csv 6.2 MB
bids_2022.csv 30.6 MB
bids_2023.csv 62.7 MB
bids_2024.csv 108.4 MB
bids_2025.csv 136.6 MB
buyers.csv 11.6 MB
suppliers.csv 35.5 MB
tenders_2022.csv 918.3 MB
tenders_2023.csv 1358.7 MB
tenders_2024.csv 1338.4 MB
tenders_2025.csv 1436.8 MB


In [62]:
files = sorted([f for f in os.listdir(path) if f.endswith('.csv')])
for f in files: 
    full = f'{path}{f}' 
    print('=' * 160)
    display(Markdown(f'### 📄 <span style="color:{color}">{f}</span>'))
    display(Markdown(f'#### • <span style="color:{color}">Структура: які колонки і їхні типи</span>'))
    con.sql(f"DESCRIBE SELECT * FROM '{full}'").show(max_rows=100)
    display(Markdown(f'#### • <span style="color:{color}">Скільки рядків</span>'))
    con.sql(f"SELECT COUNT(*) FROM '{full}'").show()
    display(Markdown(f'#### • <span style="color:{color}">Як виглядають 5 рядків</span>'))
    display(con.sql(f"SELECT * FROM '{full}' LIMIT 20").df())

### 📄 <span style="color:purple">bidders.csv</span>

#### • <span style="color:purple">Структура: які колонки і їхні типи</span>

┌────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │       column_type        │  null   │   key   │ default │  extra  │
│    varchar     │         varchar          │ varchar │ varchar │ varchar │ varchar │
├────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ bidder_id      │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ total_bids     │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ unique_tenders │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ unique_buyers  │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ first_bid_date │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ last_bid_date  │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ is_masked      │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴──────────────────────────┴─────────

#### • <span style="color:purple">Скільки рядків</span>

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        72291 │
└──────────────┘



...

┌───────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │       column_type        │  null   │   key   │ default │  extra  │
│          varchar          │         varchar          │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ tender_id                 │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ ocid                      │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ published_date            │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ tender_start_date         │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ tender_end_date           │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ award_period_start        │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ award_period_end          │ 

#### • <span style="color:purple">Скільки рядків</span>

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3610854 │
└──────────────┘



#### • <span style="color:purple">Як виглядають 5 рядків</span>

,tender_id,ocid,published_date,tender_start_date,tender_end_date,award_period_start,award_period_end,award_date,year,month,...,award_value_max,award_value_min,award_value_mean,award_value_std,award_concentration,discount_percentage_avg,discount_percentage_max,has_unsuccessful_awards,has_cancelled_awards,active_awards_count
0,e1eaec1acc9e4f3aaf8dbbb7b7dcfaeb,ocds-be6bcu-e1eaec1acc9e4f3aaf8dbbb7b7dcfaeb,2025-01-22 08:41:16.920560+02:00,NaT,NaT,NaT,NaT,2025-01-22 08:39:56.329625+02:00,2025,1,...,432.00,432.00,432.00,0.0,1.0,0.00,0.00,0,0,1
1,ef4e7e4da5ba4fababd0b9ebaeecc4df,ocds-be6bcu-ef4e7e4da5ba4fababd0b9ebaeecc4df,2025-10-29 09:06:42.457764+02:00,NaT,NaT,NaT,NaT,2025-10-29 09:06:39.668493+02:00,2025,10,...,292111.04,292111.04,292111.04,0.0,1.0,0.00,0.00,0,0,1
2,dafed2cefdcc4fc5ae2d2bfb5ddf33cd,ocds-be6bcu-dafed2cefdcc4fc5ae2d2bfb5ddf33cd,2025-09-10 13:13:23.102806+03:00,NaT,NaT,NaT,NaT,2025-09-10 13:13:20.928749+03:00,2025,9,...,2940.00,2940.00,2940.00,0.0,1.0,0.00,0.00,0,0,1
3,ff4acb26cbaf46faaaffc059ddfdbbfb,ocds-be6bcu-ff4acb26cbaf46faaaffc059ddfdbbfb,2025-10-29 11:17:10.906354+02:00,NaT,NaT,NaT,NaT,2025-10-29 11:17:09.263002+02:00,2025,10,...,8140.00,8140.00,8140.00,0.0,1.0,0.00,0.00,0,0,1
4,eaccddf19faf4afbbc4bc71e6e8abcfa,ocds-be6bcu-eaccddf19faf4afbbc4bc71e6e8abcfa,2025-01-09 16:03:48.825592+02:00,2025-01-03 21:47:04.130152+02:00,2025-01-09 09:00:00+02:00,2025-01-09 09:00:03.349862+02:00,2025-01-09 12:37:24.773265+02:00,2025-01-09 12:37:24.773265+02:00,2025,1,...,691350.00,691350.00,691350.00,0.0,1.0,11.13,11.13,0,0,1
5,af4bd8cdc6bc4fedafbcfdcdd8f0aa63,ocds-be6bcu-af4bd8cdc6bc4fedafbcfdcdd8f0aa63,2025-10-20 13:11:06.231194+03:00,NaT,NaT,NaT,NaT,2025-10-20 13:11:03.186545+03:00,2025,10,...,400.00,400.00,400.00,0.0,1.0,0.00,0.00,0,0,1
6,fcc9eaddb49b4fcba9ffacbacde31e1e,ocds-be6bcu-fcc9eaddb49b4fcba9ffacbacde31e1e,2025-02-21 11:36:28.854997+02:00,NaT,NaT,NaT,NaT,2025-02-21 11:19:53.200342+02:00,2025,2,...,87.00,87.00,87.00,0.0,1.0,0.00,0.00,0,0,1
7,4feedfabb6ed4aebbfbcd5fe726c1dec,ocds-be6bcu-4feedfabb6ed4aebbfbcd5fe726c1dec,2025-01-08 14:36:03.841066+02:00,NaT,NaT,NaT,NaT,2025-01-08 14:36:01.344228+02:00,2025,1,...,223188.00,223188.00,223188.00,0.0,1.0,0.00,0.00,0,0,1
8,9debc00f9ced4ceab2fb4fbedcfcd3ff,ocds-be6bcu-9debc00f9ced4ceab2fb4fbedcfcd3ff,2025-01-30 08:47:18.438397+02:00,2025-01-24 11:43:31.318161+02:00,2025-01-29 00:00:00+02:00,2025-01-29 00:00:11.629302+02:00,2025-01-29 08:42:03.820691+02:00,2025-01-29 08:42:03.820691+02:00,2025,1,...,35502.00,35502.00,35502.00,0.0,1.0,15.47,15.47,0,0,1
9,9fedebf3e8cd40faa8dfe75ecababeca,ocds-be6bcu-9fedebf3e8cd40faa8dfe75ecababeca,2025-04-07 09:03:13.423171+03:00,NaT,NaT,NaT,NaT,2025-04-07 08:57:32.350877+03:00,2025,4,...,3600.00,3600.00,3600.00,0.0,1.0,0.00,0.00,0,0,1


In [63]:
con.sql(f"""
    SELECT procurement_method,
           COUNT(*)                  AS n_total,
           COUNT(tender_start_date)  AS n_with_start,
           ROUND(100.0 * COUNT(tender_start_date) / COUNT(*), 1) AS pct_with_start
    FROM '{path}tenders_2025.csv'
    GROUP BY procurement_method
    ORDER BY n_total DESC
""").show()

┌────────────────────┬─────────┬──────────────┬────────────────┐
│ procurement_method │ n_total │ n_with_start │ pct_with_start │
│      varchar       │  int64  │    int64     │     double     │
├────────────────────┼─────────┼──────────────┼────────────────┤
│ limited            │ 3173816 │            0 │            0.0 │
│ selective          │  235725 │       235725 │          100.0 │
│ open               │  201313 │       201313 │          100.0 │
└────────────────────┴─────────┴──────────────┴────────────────┘



In [64]:
con.sql(f"""
    COPY (
        SELECT * FROM '{path}tenders_*.csv'
        WHERE main_cpv_code LIKE '3471%'
           OR main_cpv_code LIKE '3571%'
           OR main_cpv_code LIKE '3572%'
           OR main_cpv_code LIKE '3573%'
    ) TO '../data/processed/candidates.parquet' (FORMAT parquet)
""")